# 04 — SAM / Vicon validation with recovered original video time

Uses the existing `*_joint_angles.csv`, matching `*_sam_sync.csv` and C3D, plus camera-specific `sync_candidate.json`. **Candidates are exploratory until reviewed**. The current legacy angle exports have right-leg columns only; left-leg plots appear when genuine left-side SAM columns are available. Left = red, right = green.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise FileNotFoundError("Run Jupyter from the repository root or notebooks/")
sys.path.insert(0, str(ROOT))
from src.comparison_engine import compare_angles
from src.plotting import plot_joint_overlays, plot_joint_errors, plot_camera_comparison
DATA = ROOT / "lunge_sam_out"
RESULTS = ROOT / "results"


## Configuration
Select subject and cameras. The comparison engine recovers original video timestamps from the SAM synchronization CSV **only after verifying the legacy timestamp relationship**. Unequal row counts currently cause a safe failure; do not truncate rows to force a match.


In [ ]:
SUBJECT = "S9"
TRIAL = "lunge_01"
CAMERAS = ("cam0", "cam1")
SIDES = ("right",)  # Add "left" when the SAM angle CSV contains real l_* columns.
METHOD = "geo"
STATIC_WINDOW_MS = None  # Use only after identifying a real static interval.
ORIGINAL_TIME_CONFIRMED = True  # Verify angle rows were not resampled by the original exporter.

TRIAL_DATA = DATA / SUBJECT / TRIAL
TRIAL_RESULTS = RESULTS / SUBJECT / TRIAL
VICON_C3D = TRIAL_DATA / f"{SUBJECT.lower()}_lunge_0_1.c3d"
ANGLE_FILES = {
    camera: TRIAL_DATA / f"{SUBJECT.lower()}_lunge_{camera}_joint_angles.csv"
    for camera in CAMERAS
}
SAM_SYNC_FILES = {
    camera: TRIAL_DATA / f"{SUBJECT.lower()}_lunge_{camera}_sam_sync.csv"
    for camera in CAMERAS
}
SYNC_FILES = {
    camera: TRIAL_RESULTS / camera / "sync_candidate.json"
    for camera in CAMERAS
}
print("Vicon C3D:", VICON_C3D, VICON_C3D.exists())
for camera in CAMERAS:
    print(camera, "angles:", ANGLE_FILES[camera].exists(),
          "video timestamps:", SAM_SYNC_FILES[camera].exists(),
          "camera sync:", SYNC_FILES[camera].exists())


## Input QC
Inspect the actual available columns and timestamp counts before attempting validation. The original `time_ms_aligned` and embedded Vicon fields belong to a legacy export; they are **not** the independent reference for this run.


In [ ]:
for camera in CAMERAS:
    if not ANGLE_FILES[camera].exists() or not SAM_SYNC_FILES[camera].exists():
        continue
    sam = pd.read_csv(ANGLE_FILES[camera])
    signal = pd.read_csv(SAM_SYNC_FILES[camera])
    available = [f"{s}_{j}_flex_{METHOD}" for s in ("r", "l")
                 for j in ("hip", "knee", "ankle") if f"{s}_{j}_flex_{METHOD}" in sam]
    print(f"{camera}: angle rows={len(sam)}, sync rows={len(signal)}")
    print("  SAM columns available:", available)
    print("  Legacy aligned time present:", "time_ms_aligned" in sam)
    print("  Original sync ts present:", "ts" in signal)
    if len(sam) != len(signal):
        print("  NOTE: recovery in the installed engine currently REJECTS unequal row counts.")
    if SYNC_FILES[camera].exists():
        import json
        sync_meta = json.loads(SYNC_FILES[camera].read_text())
        print("  Candidate offset (ms):", sync_meta.get("shift_ms"),
              "status:", sync_meta.get("status"))


## Compare cameras and available sides
This calls the modified `src.comparison_engine.compare_angles` with the *existing* SAM angle CSV, its original-time sync export, and the Vicon C3D. Camera candidates are exploratory. A failure on one camera does not stop the other.


In [ ]:
results = []
aligned_by_camera = {}
failed = []
for camera in CAMERAS:
    sam_csv = ANGLE_FILES[camera]
    sam_time_csv = SAM_SYNC_FILES[camera]
    sync_json = SYNC_FILES[camera]
    if not all(p.exists() for p in (sam_csv, sam_time_csv, sync_json, VICON_C3D)):
        print(f"SKIP {camera}: one or more input files missing")
        continue
    sam_headers = pd.read_csv(sam_csv, nrows=0).columns
    for side in SIDES:
        prefix = "l" if side == "left" else "r"
        if not any(f"{prefix}_{j}_flex_{m}" in sam_headers
                   for j in ("hip", "knee", "ankle") for m in ("geo", "euler", "smooth")):
            print(f"SKIP {camera}/{side}: no genuine SAM {side}-leg columns")
            continue
        out = TRIAL_RESULTS / camera / "validation" / side
        try:
            metrics = compare_angles(
                angle_csv=sam_csv, output_dir=out,
                sam_sync_csv=sam_time_csv,
                sync_json=sync_json, vicon_c3d=VICON_C3D,
                original_time_confirmed=ORIGINAL_TIME_CONFIRMED,
                subject=SUBJECT, trial=TRIAL, camera=camera, side=side,
                static_window_ms=STATIC_WINDOW_MS,
                diagnostic_shift_ms=None, plot=False,
            )
        except (ValueError, KeyError, FileNotFoundError) as e:
            print(f"FAILED {camera}/{side}: {e}")
            failed.append({"camera": camera, "side": side, "error": str(e)})
            continue
        results.append(metrics)
        side_frame = pd.read_csv(out / "aligned_angles.csv")
        if camera not in aligned_by_camera:
            aligned_by_camera[camera] = side_frame
        else:
            for col in side_frame.columns:
                if col.startswith(prefix + "_vicon_"):
                    aligned_by_camera[camera][col] = side_frame[col].to_numpy()

if results:
    summary = pd.concat(results, ignore_index=True)
    display(summary[["camera", "side", "joint", "method", "n_samples",
                     "rmse_raw_deg", "bias_deg", "pearson_r", "rom_error_deg"]].round(2))
    target = TRIAL_RESULTS / "validation_metrics_bilateral.csv"
    summary.to_csv(target, index=False)
    print("Saved", target)
else:
    print("No validation metrics generated; see failures and missing inputs above.")
if failed:
    display(pd.DataFrame(failed))


## Bilateral plots
Solid line = SAM, dashed line = Vicon; green = right, red = left. Each camera uses its **own** independently reviewed synchronization offset.


In [ ]:
for camera, frame in aligned_by_camera.items():
    fig = plot_joint_overlays(frame,
        title=f"{SUBJECT} {TRIAL} {camera}: bilateral angles", method=METHOD)
    display(fig)
    plt.close(fig)
    fig = plot_joint_errors(frame,
        title=f"{SUBJECT} {TRIAL} {camera}: bilateral errors", method=METHOD)
    display(fig)
    plt.close(fig)
if len(aligned_by_camera) >= 2:
    fig = plot_camera_comparison(aligned_by_camera, joint="knee", method=METHOD)
    display(fig)
    plt.close(fig)


## Interpretation and limitations
- `sync_candidate.json` is still **review**, not accepted synchronization.
- The current SAM `joint_angles.csv` files may only contain genuine right-leg angles; missing left-side values must not be fabricated.
- Original video time is recovered by comparing original `sam_sync.csv` timestamps against legacy aligned timestamps. The current comparison engine raises an error if row counts differ; do not truncate files or force row-wise matching.
- The user must verify that the historical angle export did not interpolate/resample the SAM values.
- C3D marker-based hip/ankle angles require anatomical and sign-convention QC before reporting clinical accuracy.
- `plotting.py` must be the updated bilateral version that accepts `method=`.
- No joint-derived timing adjustment or neutral calibration is applied in this notebook.
